# Does the Negative Result Survive a REAL Tree? (Phase D — the GO/NO-GO)

Every number in the paper so far was measured on a **flat FAISS index** (`granularity_sweep.build_flat_retriever`) — no RAPTOR tree is ever built — while the paper claims to study *hierarchical* leaf granularity. The dry run priced the fix: **$0.09 at n=50, $0.78 for all 416 docs**. So the "prohibitive on a free-tier budget" justification for the flat proxy costs less than a cup of coffee to remove.

This notebook runs the sweep **both ways on the same documents** and answers two questions:

1. **Does H2's negative result survive?** Flat says the best fixed size is 200 tokens with a per-doc oracle 9.8% above it. Does a real tree agree?
2. **How confounded is leaf size with tree depth?** The dry run measured `layers` falling 2.0 → 0.8 across the size grid. **0.8 means some documents build NO tree at all** — `construct_tree` stops once a layer is down to 11 nodes, and a short paper at 400-token leaves never clusters. Collapsed-tree retrieval over a 0-layer tree *is* flat retrieval.

> **This needs a real API key.** The dry run could fake summaries because it only counted calls. Here the summaries *are* the tree's content — they get embedded, clustered, and retrieved. An extractive stand-in would produce meaningless coverage.

### Kaggle setup
1. **Settings → Internet → On**
2. **Settings → Accelerator → GPU T4**
3. **Add-ons → Secrets** → add `OPENROUTER_API_KEY`
4. Load credit at openrouter.ai (min $10). The run itself spends **~$0.09** — the free tier's 1000 req/day would take 3.2 days for the same thing, and the $10 that unlocks it costs 100x the run.

Expect ~2 hours. It is resumable: re-run cell 5 after a session timeout and it picks up where it stopped.


In [ ]:
# 1) Code + preflight. Heavy deps are imported LAZILY inside RAPTOR, so a light
#    import proves nothing — check them for real or cell 5 dies confusingly.
!git clone -b ckraptor https://github.com/MissLostCodes/raptor.git
%cd raptor
!pip install -q -r requirements-colab.txt

from experiments.granularity_sweep import run_sweep, build_tree_retriever, best_size_per_doc

import importlib
missing = []
for mod in ['torch', 'sentence_transformers', 'umap', 'faiss', 'sklearn', 'tiktoken']:
    try:
        importlib.import_module(mod)
    except Exception as e:
        missing.append(f'{mod:<22} {type(e).__name__}: {e}')
if missing:
    print('PREFLIGHT FAILED \u274c\n')
    for m in missing:
        print('   ', m)
    print('\nKaggle: Internet On, then Run -> Restart Session and re-run.')
    raise SystemExit('preflight failed')

print('heavy stack present \u2705')

In [ ]:
# 2) Config + secrets + persisted output.
N_DOCS  = 50                             # the paper's H1/H2 cohort. ~$0.09, ~2h.
SIZES   = [50, 100, 150, 200, 300, 400]  # the paper's sweep grid
BUDGET  = 2000                           # retrieval token budget (paper's setting)

# PAID model id: no ':free' suffix. ':free' is capped at 1000 req/day (3.2 days for
# this run); paid is $0.03/$0.15 per M tokens and finishes in one session for ~$0.09.
MODEL = 'openai/gpt-oss-120b'

import os

def api_key():
    try:
        from google.colab import userdata
        return userdata.get('OPENROUTER_API_KEY')
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('OPENROUTER_API_KEY')
    except Exception:
        pass
    return os.environ.get('OPENROUTER_API_KEY')

key = api_key()
if not key:
    raise SystemExit('Need OPENROUTER_API_KEY (Kaggle: Add-ons -> Secrets). '
                     'A real tree needs real summaries — no offline mode here.')
os.environ['OPENROUTER_API_KEY'] = key

def run_dir():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        return '/content/drive/MyDrive/raptor_runs'
    except Exception:
        pass
    if os.path.isdir('/kaggle/working'):
        return '/kaggle/working/raptor_runs'
    return 'raptor_runs'

RUN_DIR = run_dir()
os.makedirs(RUN_DIR, exist_ok=True)
# SEPARATE files per mode. The resume key is (doc_id, size) and carries no mode, so
# sharing one path would resume a tree run from flat records. run_sweep refuses, but
# distinct names are the real fix.
FLAT_OUT = os.path.join(RUN_DIR, f'sweep_flat_n{N_DOCS}.json')
TREE_OUT = os.path.join(RUN_DIR, f'sweep_tree_n{N_DOCS}.json')
print('FLAT_OUT =', FLAT_OUT)
print('TREE_OUT =', TREE_OUT)

In [ ]:
# 3) Load the QASPER cohort (same split + loader the paper uses).
from experiments.datasets import get_loader
import statistics

docs = get_loader('qasper').load(limit=N_DOCS)
lens = [len(d.text.split()) for d in docs]
nq = sum(len(d.questions) for d in docs)
print(f'{len(docs)} docs | {nq} questions | median {statistics.median(lens):.0f} words')

In [ ]:
# 4) FLAT baseline (no LLM, ~15 min). Reproduces the paper's proxy on THESE docs so
#    the tree comparison below is apples-to-apples in one session.
from tqdm.auto import tqdm

flat_records = run_sweep(
    docs, SIZES, budget=BUDGET, out_path=FLAT_OUT,
    progress=lambda m: tqdm.write(m),
)
print(f'\nflat: {len(flat_records)} (doc, size) records')

In [ ]:
# 5) TREE sweep — the real thing. Builds a RAPTOR tree per (doc, size): clusters +
#    LLM summaries + collapsed-tree retrieval. Scoring stays LLM-free.
#    ~3200 summarization calls, ~$0.09, ~2h. RESUMABLE: re-run this cell after a
#    timeout and it skips completed (doc, size) pairs.
from experiments.config import ExperimentConfig
from experiments.models import build_models

cfg = ExperimentConfig(model=MODEL, cache_dir=os.path.join(RUN_DIR, '.llm_cache'))
summarization_model, _ = build_models(cfg)   # DiskCache-backed: restarts are free

tree_records = run_sweep(
    docs, SIZES, budget=BUDGET, out_path=TREE_OUT,
    summarization_model=summarization_model,   # <- this is what makes it hierarchical
    progress=lambda m: tqdm.write(m),
)
print(f'\ntree: {len(tree_records)} (doc, size) records')

In [ ]:
# 6) THE CONFOUND. Tree depth per leaf size — is leaf size even an independent variable?
from collections import defaultdict
import statistics as st

by_size = defaultdict(list)
for r in tree_records:
    if r.get('n_layers') is not None:
        by_size[r['size']].append(r)

print(f"{'leaf size':>10}{'mean layers':>13}{'zero-layer docs':>17}{'mean leaves':>13}")
print('-' * 53)
for s in sorted(by_size):
    rs = by_size[s]
    layers = [r['n_layers'] for r in rs]
    zero = sum(1 for l in layers if l == 0)
    leaves = [r['n_chunks'] for r in rs]
    print(f"{s:>10}{st.mean(layers):>13.2f}"
          f"{f'{zero}/{len(rs)} ({100*zero/len(rs):.0f}%)':>17}{st.mean(leaves):>13.1f}")

total_cells = sum(len(v) for v in by_size.values())
total_zero = sum(1 for v in by_size.values() for r in v if r['n_layers'] == 0)
print(f"\nZero-layer cells: {total_zero}/{total_cells} ({100*total_zero/total_cells:.1f}%) "
      f"— these ran FLAT retrieval while labelled hierarchical.")

In [ ]:
# 7) THE VERDICT. H1 flat vs tree: does the negative result survive a real tree?
def h1(records, label):
    per_size = defaultdict(list)
    for r in records:
        if r.get('mean_evidence_coverage') is not None:
            per_size[r['size']].append(r['mean_evidence_coverage'])
    means = {s: st.mean(v) for s, v in per_size.items()}
    best = max(means, key=means.get)

    oracle = best_size_per_doc(records)
    by_doc = defaultdict(dict)
    for r in records:
        if r.get('mean_evidence_coverage') is not None:
            by_doc[r['doc_id']][r['size']] = r['mean_evidence_coverage']
    orc = st.mean([by_doc[d][s] for d, s in oracle.items() if s in by_doc[d]])

    gain = 100 * (orc - means[best]) / means[best]
    print(f'\n{label}')
    print(f"  per-size coverage: " + '  '.join(f'{s}:{means[s]:.3f}' for s in sorted(means)))
    print(f"  best fixed size  : {best} tok @ {means[best]:.4f}")
    print(f"  per-doc oracle   : {orc:.4f}  (+{gain:.1f}% rel headroom)")
    return best, means[best], orc

h1(flat_records, 'FLAT (the paper\'s current proxy)')
h1(tree_records, 'TREE (real RAPTOR)')

## How to read this

**Cell 6 — the confound.** If `zero-layer docs` is non-zero at coarse sizes, those cells ran flat retrieval while being reported as hierarchical. That is the measured refutation of the paper's central methodological premise (`main.tex:100-102`):

> *"The leaf chunker is therefore a clean, isolatable variable inside an otherwise-frozen pipeline"*

It isn't. Leaf size → leaf count → cluster count → tree depth is a causal chain, so leaf size and architecture cannot be varied independently. That is the paper's new headline, and it *explains* the three negative results instead of merely adding a fourth.

Pre-empt the obvious reviewer attack: *"that's just `reduction_dimension=10`, change the config."* Partly fair — the **threshold** is a config choice, but the **coupling** is structural (fewer leaves → fewer clusters → shallower tree, and UMAP needs a minimum point count regardless). Claim exactly that, no more.

**Cell 7 — the verdict.** Three outcomes, all publishable:

| tree result | meaning |
|---|---|
| best size ≈ 200, headroom ≈ 10% | negative result **survives**; the paper's claims now match its measurements |
| best size shifts | the "chunk-size optimum" was partly a **tree-depth optimum** — a stronger finding |
| headroom collapses | leaf size matters *less* in a tree; the flat proxy overstated the whole premise |

**Next if this clears:** re-run at `N_DOCS=None` (all 416, ~$0.78) and feed `tree_records` to `headroom_decomposition` for the nested-oracle ledger — the `per_question` field is already carried through, so that path needs no changes.
